# 06-1 · DACON 236036 기계 고장 진단 (팬 소음) — **미채택 기록**

| | |
|---|---|
| **목적** | 14.3GB 를 받아 놓고 **왜 안 쓰는지**를 실측으로 남긴다 |
| **input** | `data/raw/DACON 236036 .../train.csv · test.csv · sample_submission.csv` |
| **output** | `data-analysis/data/processed/dacon_fan_sound.parquet`<br>`data-analysis/reports/dacon_fan_sound_decisions.json` |

> 🔢 **번호에 `-1` 이 붙은 이유**: `90_merge` 에 **참여하지 않는 독립 데이터셋**이다.
> 다른 데이터셋과 공유하는 join key 가 없다.

> ⛔ **wav 5.1MB × 2,793개는 열지 않는다.** 이 노트북은 **메타데이터 CSV 3개만** 본다 —
> 미채택 근거는 라벨 구조에서 나오지 결국 오디오에서 나오지 않기 때문이다.

In [ ]:
import sys
sys.path.insert(0, "..")
import struct
import pandas as pd
from common import RAW, profile, save_decisions, save_processed

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


## 1. 개요

| 항목 | 값 |
|---|---|
| 출처 | [dacon.io / 236036](https://dacon.io/competitions/official/236036/data) |
| 수집일 | 2026-08-12 |
| 규모 | **14.3GB** — train 1,279 wav(6.5GB) · test 1,514 wav(7.8GB) |
| 과제 | 팬(fan) 소음 **비지도 이상탐지** |

**결론 먼저: 라벨된 고장 샘플이 0건이라 못 쓴다.** 아래에서 직접 확인한다.

In [ ]:
D = RAW / "DACON 236036 월간 데이콘 기계 고장 진단 AI 경진대회"
train = pd.read_csv(D / "train.csv")
test = pd.read_csv(D / "test.csv")
sub = pd.read_csv(D / "sample_submission.csv")
for n, d in [("train.csv", train), ("test.csv", test), ("sample_submission.csv", sub)]:
    print(f"{n:24} {d.shape[0]:>6,}행 × {d.shape[1]}열   {list(d.columns)}")

## 2. 로드 + 기초 통계

In [ ]:
profile(train)

In [ ]:
train.head(5)

## 3. EDA

### 3-1. 🚨 `train.csv` 의 `LABEL` 이 전부 0 이다

In [ ]:
print("train LABEL 분포 —")
print(train["LABEL"].value_counts().to_string())
print()
print(f"고유값: {train['LABEL'].unique().tolist()}")
print(f"→ {len(train):,}행 전부 0(정상). **고장 샘플이 한 건도 없다.**")
print()
print("지도학습 분류가 아니라 비지도 이상탐지(정상만 학습 → 이상 판정) 과제다.")

### 3-2. 🚨 `test.csv` 에는 `LABEL` 열 자체가 없다

In [ ]:
print("train 컬럼:", list(train.columns))
print("test  컬럼:", list(test.columns))
print()
print("LABEL in test?", "LABEL" in test.columns)
print()
print("sample_submission LABEL 분포 (제출 양식) —")
print(sub["LABEL"].value_counts().to_string())
print("→ 전부 0 placeholder. 정답은 공개되지 않는다.")
print()
print("⛔ 결론: 라벨된 고장 샘플 0건. '실제 고장 사례 확보' 목적으로도 쓸 수 없다.")

### 3-3. `FAN_TYPE` 분포 — 유일하게 변별력 있는 축

In [ ]:
both = pd.concat([train.assign(split="train"), test.assign(split="test")], ignore_index=True)
print(pd.crosstab(both["split"], both["FAN_TYPE"], margins=True).to_string())
print()
print("→ FAN_TYPE 은 0 과 2 뿐이다 (1 은 없다). 대체로 균형.")

### 3-4. wav 형식 — 표준 `wave` 모듈로 못 읽는다

파일 **1개만** 헤더 44바이트를 직접 파싱한다. 오디오 데이터는 읽지 않는다.

In [ ]:
one = sorted((D / "train").glob("*.wav"))[0]
b = one.read_bytes()[:80]
i = 12
while i < len(b) - 8:
    cid, sz = b[i:i + 4], struct.unpack("<I", b[i + 4:i + 8])[0]
    if cid == b"fmt ":
        fmt, ch, sr, bps, align, bits = struct.unpack("<HHIIHH", b[i + 8:i + 24])
        n = one.stat().st_size
        print(f"파일        : {one.name}  ({n:,} bytes)")
        print(f"audio_format: {fmt}  (1=PCM, 3=IEEE float)")
        print(f"채널        : {ch}   ← 마이크 어레이")
        print(f"샘플레이트  : {sr:,} Hz")
        print(f"비트        : {bits} bit")
        print(f"길이        : {(n - 44) / align / sr:.1f} 초")
        break
    i += 8 + sz + (sz % 2)
print()
print("⚠ audio_format=3 (32bit float) 이라 파이썬 표준 wave 모듈은")
print("  'unknown format: 3' 로 실패한다. soundfile/librosa 가 필요하다.")

## 4. 전처리 — 메타데이터만 통합

In [ ]:
df = both.copy()
df["LABEL"] = df["LABEL"].astype("Int64")        # test 는 <NA>
df["FAN_TYPE"] = df["FAN_TYPE"].astype("Int64")
df["라벨있음"] = df["LABEL"].notna()
df["채택"] = False                                # 이 데이터셋은 미채택이다 (기록용)
print(f"정제 후: {df.shape[0]:,}행 × {df.shape[1]}열")
print(df.groupby("split")["라벨있음"].agg(["size", "sum"]).to_string())

## 5. 결정 로그

In [ ]:
column_decisions = [
    {"column": "SAMPLE_ID", "used": True, "reason": "행 식별자"},
    {"column": "SAMPLE_PATH", "used": True, "reason": "wav 경로 — 실제로 열지는 않았다"},
    {"column": "FAN_TYPE", "used": True, "reason": "0/2 두 종류. 이 데이터셋에서 유일하게 변별력 있는 축"},
    {"column": "LABEL", "used": False,
     "reason": "🚨 train 1,279행 전부 0 이고 test 엔 열 자체가 없다 — 학습 신호가 0"},
    {"column": "split", "used": True, "reason": "train/test 구분 (이 노트북에서 만든 파생)"},
]
print(f"컬럼 결정 {len(column_decisions)}건")

In [ ]:
save_decisions(
    "dacon_fan_sound",
    column_decisions,
    source_url="https://dacon.io/competitions/official/236036/data",
    collected_at="2026-08-12",
    adopted=False,
    size_on_disk="14.3GB",
    row_count_raw=int(len(train) + len(test)),
    wav_files={"train": 1279, "test": 1514},
    wav_format="8ch · 16,000Hz · 32bit IEEE float · 10.0s · 5.1MB/file",
    generated_columns=["split", "라벨있음", "채택"],
    rejection_reasons=[
        "① train.csv 의 LABEL 이 1,279행 전부 0(정상) — 지도학습 분류가 아니라 비지도 이상탐지",
        "② test.csv 에 LABEL 열이 없고 정답 미공개 → 라벨된 고장 샘플 0건. "
        "'실제 고장 사례 확보' 목적으로도 못 쓴다",
        "③ 진단 경로가 에러코드 텍스트 기반이고(D1) MCP 도구 15종에 오디오 입력이 없다",
        "④ 대상이 팬 소리 — MaintQ 는 인버터(model enum 2종 고정, D6·D13)와 호스트 설비",
        "⑤ 14.3GB 는 git 불가 · 멜스펙트로그램 등 특징 추출 파이프라인이 통째로 추가된다",
    ],
    caveats=[
        "32bit float 라 표준 wave 모듈로 못 읽는다 (unknown format: 3)",
        "붙는다면 백로그 P10 — 단 현재 P10 입력이 error_history(D29)라 입력 축 신설이 선행",
    ],
)

## 6. 정제 데이터 저장 — 메타만 (wav 는 제외)

In [ ]:
save_processed("dacon_fan_sound", df)